In [1]:
from Utils import TempRel_Utils
import torch
from transformers import RobertaForSequenceClassification, RobertaTokenizerFast, TrainingArguments, Trainer, DataCollatorWithPadding
from Reader import obtain_combined_dataset

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    device = torch.device('cuda')
print("Current Device:", torch.cuda.current_device(), torch.cuda.get_device_name(torch.cuda.current_device()))

CUDA available: True
Current Device: 0 NVIDIA GeForce RTX 4070 Ti


In [3]:
datasets, label_list, label2id, id2label = obtain_combined_dataset(["TempEval3", "MATRES", "TBDense"], "TempRel")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Stringifying the column:   0%|          | 0/172689 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/172689 [00:00<?, ? examples/s]

In [ ]:
# datasets["train"].to_json(".\\cleandata\\combined\\TempRel\\train.json")
# datasets["test"].to_json(".\\cleandata\\combined\\TempRel\\test.json")
# datasets["eval"].to_json(".\\cleandata\\combined\\TempRel\\eval.json")

In [4]:
model_name = 'roberta-base'
tokenizer = RobertaTokenizerFast.from_pretrained(model_name, add_prefix_space=True)
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=len(label_list), label2id=label2id, id2label=id2label)

seps = ["<e1>", "</e1>", "<e2>", "</e2>", "<e>", "</e>", "<timex", "</timex>", "TIMEVAL=", "TYPE=DATE>", "TYPE=TIME>", "TYPE=DURATION>", "TYPE=SET>", "TYPE=UNKOWN"]
tokenizer.add_tokens(seps)
model.resize_token_embeddings(len(tokenizer))
utils = TempRel_Utils(tokenizer, label2id, id2label)

d:\GeoTKG\venv\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at roberta-base were not used when initializing RobertaForSequenceClassification: ['lm_head.layer_norm.weight', 'lm_head.bias', 'lm_head.dense.weight', 'lm_head.dense.bias', 'lm_head.layer_norm.bias']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassifica

In [5]:
datasets = utils.tokenize_datasets(datasets)

Map:   0%|          | 0/34538 [00:00<?, ? examples/s]

Map:   0%|          | 0/124335 [00:00<?, ? examples/s]

Map:   0%|          | 0/13816 [00:00<?, ? examples/s]

Map:   0%|          | 0/34538 [00:00<?, ? examples/s]

Map:   0%|          | 0/124335 [00:00<?, ? examples/s]

Map:   0%|          | 0/13816 [00:00<?, ? examples/s]

In [6]:
datasets

DatasetDict({
    test: Dataset({
        features: ['tokens', 'label', 'input_ids', 'attention_mask'],
        num_rows: 34538
    })
    train: Dataset({
        features: ['tokens', 'label', 'input_ids', 'attention_mask'],
        num_rows: 124335
    })
    eval: Dataset({
        features: ['tokens', 'label', 'input_ids', 'attention_mask'],
        num_rows: 13816
    })
})

In [ ]:
#datasets = datasets.remove_columns("label")

In [7]:
training_args = TrainingArguments(
    output_dir="./results/TempRel2",
    logging_dir="./logs",
    evaluation_strategy="steps",
    save_strategy="steps",
    logging_steps=100,
    save_steps=15000,
    eval_steps=15000,
    num_train_epochs=10,
    save_total_limit=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    metric_for_best_model="f1"
)

In [8]:
event_timex_temprel = Trainer(
    model=model,
    args=training_args,
    compute_metrics=utils.compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt"),
    tokenizer=tokenizer,
    train_dataset=datasets["train"],
    eval_dataset=datasets["eval"],
)

In [ ]:
event_timex_temprel.train()

The following columns in the training set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: tokens. If tokens are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.
d:\GeoTKG\venv\Lib\site-packages\transformers\optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 124335
  Num Epochs = 10
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 77710
  Number of trainable parameters = 124661767


  0%|          | 0/77710 [00:00<?, ?it/s]

You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


{'loss': 0.9912, 'learning_rate': 4.993565821644576e-05, 'epoch': 0.01}


In [ ]:
event_timex_temprel.save_model("./results/TempRel2")
event_timex_temprel.tokenizer.save_pretrained("./results/TempRel2")

In [ ]:
event_timex_temprel.evaluate(datasets["test"])